In [2]:
import os
import json
from prettytable import PrettyTable
import statistics

# 假设 evaluate_output_folder 在 config_task 中定义
# 如果你直接运行，请确保此路径正确或手动定义它
from config_task import evaluate_output_folder
# evaluate_output_folder = "./outputs" # 请根据实际路径修改

dataset_info = (
    'UAVTOD_DJI0824Part5_0',
    'UAVTOD_DJI0824Part5_1',
    'UAVTOD_DJI0827Part1_0',
    'UAVTOD_DJI0827Part1_1',
    'UAVTOD_DJI0828Part1_0',
    'UAVTOD_DJI0828Part1_1',
    'UAVTOD_DJI0830Part3_0',
    'UAVTOD_DJI0830Part3_1',
    'UAVTOD_DJI0833Part4_0',
    'UAVTOD_DJI0833Part4_1',
    'UAVTOD_DJI0839Part3_0',
    'UAVTOD_DJI0839Part3_1'
)

criteria_name = ("AR_es", "AR_es_move", "AR_et", "AR_et_move", "FPS")

model_names = ['yoloft-L', 'FracSTMD', 'vSTMD_F', ]


def collect_result():
    # 动态创建字典存储结果
    for criterion in criteria_name:
        exec(f'{criterion}_dict = dict()')

    for datasetName in dataset_info:
        file_path = os.path.join(evaluate_output_folder, f'{datasetName}.json')
        if not os.path.exists(file_path):
            continue
            
        with open(file_path, 'r') as f:
            _data = json.load(f)

        for modelName in model_names:
            for criterion in criteria_name:
                exec(f'{criterion}_dict.setdefault(datasetName, dict())')
                try:
                    value = _data[modelName][criterion]
                    exec(f'{criterion}_dict[datasetName][modelName] = value')
                except KeyError:
                    exec(f'{criterion}_dict[datasetName][modelName] = "-"')

    # 将汇总结果保存到 JSON
    with open('XS-VID_result.json', 'w') as json_file:
        json.dump({
            criterion: eval(f'{criterion}_dict')
            for criterion in criteria_name
        }, json_file, indent=4)


def show_table():


    with open('XS-VID_result.json', 'r') as json_file:
        results = json.load(json_file)

    # --- 初始化汇总表数据结构 ---
    # summary_stats 结构: { "model_name": [mean_of_crit1, mean_of_crit2, ...] }
    summary_stats = {model: [] for model in model_names}

    for criterion in criteria_name:
        table = PrettyTable()
        table.field_names = ["Dataset"] + model_names
        
        criterion_dict = results[criterion]
        
        # 用于存储当前 Criterion 下每个模型的所有数值，以便计算 Mean
        model_val_collector = {model: [] for model in model_names}

        for dataset_name in dataset_info:
            row = [dataset_name]
            
            for model in model_names:
                value = criterion_dict[dataset_name].get(model, 0) # 使用 get 防止键不存在
                row.append(value)
                
                # 收集数值用于计算平均值
                if isinstance(value, (int, float)):
                    model_val_collector[model].append(value)
            
            table.add_row(row)

        # --- 计算并添加 Mean 行 ---
        mean_row = ["Mean"]
        for model in model_names:
            vals = model_val_collector[model]
            avg = round(statistics.mean(vals), 4) if vals else 0
            mean_row.append(avg)
            
            # 将该平均值存入总表结构
            summary_stats[model].append(avg)
        
        table.add_row(mean_row)

        print(f"=== {criterion} ===")
        print(table)
        print("\n")
        

    # --- 生成最终的总表 (Only Means) ---
    summary_table = PrettyTable()
    summary_table.field_names = ("Model", ) + criteria_name
    
    for model in model_names:
        # 将模型名和它在各个指标下的平均值组合成一行
        summary_table.add_row([model] + summary_stats[model])

    print("=== Summary Table (Average Performance) ===")
    print(summary_table)

    # 如果需要 LaTeX 格式，可以直接调用：
    print("=== Summary Table in LaTeX Format ===")
    print(summary_table.get_latex_string())
    print("\n")



if __name__ == '__main__':
    collect_result() 
    show_table()

=== AR_es ===
+-----------------------+---------------------+---------------------+---------------------+
|        Dataset        |       yoloft-L      |       FracSTMD      |       vSTMD_F       |
+-----------------------+---------------------+---------------------+---------------------+
| UAVTOD_DJI0824Part5_0 |  0.6722205392305362 | 0.21084519842471977 |  0.5649803089972736 |
| UAVTOD_DJI0824Part5_1 |  0.7029023746701847 | 0.26437994722955144 |  0.6147757255936676 |
| UAVTOD_DJI0827Part1_0 | 0.37572254335260113 |  0.3495409724583475 |  0.7252635158109486 |
| UAVTOD_DJI0827Part1_1 |  0.4178136059771923 | 0.23515532835233977 |  0.6415650806134486 |
| UAVTOD_DJI0828Part1_0 | 0.47015873015873016 | 0.12825396825396826 |  0.5425396825396825 |
| UAVTOD_DJI0828Part1_1 |  0.4284188034188034 | 0.11217948717948718 |  0.5833333333333334 |
| UAVTOD_DJI0830Part3_0 | 0.21298701298701297 |  0.2961038961038961 | 0.37922077922077924 |
| UAVTOD_DJI0830Part3_1 |  0.8243243243243243 | 0.2432432432432432